# Credit Card Fraud Detection

In [ ]:
import glob
# Colab bootstrap: make credit_fraud_pack importable on the remote runtime.
# Does nothing when the notebook runs locally.
import os, sys

ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    REPO_URL = "https://github.com/Linamandla-Mzamane/Credit-Card-Fraud-Detection"
    REPO_DIR = "Credit-Card-Fraud-Detection"
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL}
    if os.path.basename(os.getcwd()) != REPO_DIR:
        %cd {REPO_DIR}
    sys.path.insert(0, os.path.abspath("src"))
    !pip install -q kagglehub

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from credit_fraud_pack.pipeline import build_model_pipeline
from credit_fraud_pack.evaluate import evaluate_classifier, plot_confusion_matrix
from credit_fraud_pack.data import load_raw_data
from credit_fraud_pack.config import (PCA_COLS,
                                      FEATURE_COLS,
                                      TARGET_COL,
                                      RANDOM_SEED,
                                      TEST_SIZE)

In [ ]:
# Visualisations style variables
COLOR_LEGIT = "#0072B2"
COLOR_FRAUD = "#D55E00"
CLASS_PALETTE = {0: COLOR_LEGIT, 1: COLOR_FRAUD}

sns.set_theme(style="whitegrid")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.dpi"] = 100

pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 120)

## Load Data

In [ ]:
if ON_COLAB:
    import getpass, shutil, glob, kagglehub
    from credit_fraud_pack.config import RAW_DATA_DIR

    os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ")
    os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle key: ")

    cache_dir = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    print("kagglehub cache:", cache_dir)

    matches = glob.glob(f"{cache_dir}/**/creditcard.csv", recursive=True)
    print("found:", matches)

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(matches[0], RAW_DATA_DIR / "creditcard.csv")
    print("copied to:", RAW_DATA_DIR / "creditcard.csv")

In [ ]:
# Load raw data
df = load_raw_data()
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

All 31 features in the dataset are numeric and the dataset takes up about 67.4 MBs, which is small enough to work with in pandas without chunking. There is no mix of data types that need cleaning.

In [ ]:
# Check for missing values
num_missing = df.isnull().sum().sum()
print(f"Total missing values: {num_missing}")

In [ ]:
# Check for duplicates
num_duplicates = df.duplicated().sum()
duplicate_share = num_duplicates / len(df)
print(f"Exact duplicate rows: {num_duplicates} ({duplicate_share:.3%} of the data)")

There are zero missing values, however 1,081 exact duplicate rows (~0.38% of the data). Before dropping any duplicates the class that they fall into will be checked, this is because dropping duplicates will shrink the already small set of fraud data.

In [ ]:
print(df.loc[df.duplicated(), "Class"].value_counts())

df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

## Class Balance

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index() * 100
pd.DataFrame({"count": class_counts, "percent": class_pct.round(3)})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], class_counts, color=[COLOR_LEGIT, COLOR_FRAUD])

ax.set_yscale("log")
ax.set_ylabel("Number of transactions (log scale)")
ax.set_title("Class balance: legitimate vs. fraudulent transactions")

for bar, count, pct in zip(bars, class_counts, class_pct):
    ax.annotate(f"{count:,}\n({pct:.3f}%)",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom")

ax.set_ylim(top=class_counts.max() * 4)
plt.tight_layout()
plt.show()

**Observation:** Fraud makes up only 0.167% of transactions (473 out of 283,726), which is about 1 in every 600.

- **Accuracy isn't the most important metric:** A model that always predicts "not fraud" would score 99.8% accuracy while catching zero fraud, so evaluation should rely on precision, recall, F1 and AUPRC instead.
- **The train test split must be stratified on "Class":** A random split could easily leave the test set with too few fraud cases to evaluate against, it could even leave the test set with zero fraud cases.
- **The class imbalance itself needs direct treatment:** Class weighing or resampling such as SMOTE will be used because most classifiers default to optimising overall accuracy and will otherwise learn to mostly ignore the minority class.

## Why `Time` and `Amount` need different treatment from `V1` - `V28`

In [ ]:
v_cols = PCA_COLS
scale_summary = df[v_cols + ["Time", "Amount"]].agg(["mean", "std"]).T
scale_summary.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(v_cols, df[v_cols].std(), color=COLOR_LEGIT)
ax.set_ylabel("Standard Deviation")
ax.set_title("Spread of each PCA component (V1 - V28)")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

`V1`-`V28` all have a mean near 0 and a standard deviation that is at most a little bit above 1. The standard deviation decreases fairly steadyily from `V1`-`V28`, which makes sense given that the transaction features where already transformed via PCA.

`Time` and `Amount` on the other hand, were not part of the PCA step and sit on completely different scales, with a standard deviation of 47,481 (seconds) and 250, respectively. Distance and gradient based models would let `Time` and `Amount` influence the prediction much more purely because of scale. This is why it is necessary to scale the features using the StandardScaler.

## Transaction amount

In [ ]:
df["Amount"].describe()

In [ ]:
# Same summary, split by class, to see whether fraud looks different
summary_cols = ["count", "mean", "50%", "std", "max"]
df.groupby("Class")["Amount"].describe()[summary_cols]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
class_labels = [(0, COLOR_LEGIT, "Legitimate"), (1, COLOR_FRAUD, "Fraud")]
for cls, color, label in class_labels:
    ax.hist(np.log1p(df.loc[df["Class"] == cls, "Amount"]), bins=50,
            density=True, alpha=0.55, color=color, label=label)

ax.set_xlabel("log(1 + Amount)")
ax.set_ylabel("Density")
ax.set_title("Transaction amount distribution by class (log scaled)")
ax.legend(title="Class")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=df, x="Class", y="Amount", hue="Class", palette=CLASS_PALETTE,
            showfliers=False, legend=False, ax=ax)

ax.set_yscale("log")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Legitimate", "Fraud"])
ax.set_ylabel("Amount (log scale)")
ax.set_title("Amount by class (outliers hidden, log scale)")
plt.tight_layout()
plt.show()

**Observation:** The median of fraudulent transactions is smaller than the median of the legitimate ones.

## Transaction Timing

In [ ]:
# Time in seconds since the first transaction over a 2-day window
df["hour_in_period"] = df["Time"] // 3600       # 0-47
df["hour_of_day"] = df["hour_in_period"] % 24   #0-23, both days combined
df["hour_in_period"].max(), df["hour_of_day"].max()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df["hour_in_period"].value_counts().sort_index().plot(
    kind="line", ax=ax, color=COLOR_LEGIT, marker="o", markersize=3
)

ax.set_xlabel("Hours since the first transaction")
ax.set_ylabel("Transaction volume")
ax.set_title("Transaction volume across the two day collection window")
plt.tight_layout()
plt.show()

In [ ]:
fraud_rate_by_hour = df.groupby("hour_of_day")["Class"].mean() * 100
overall_rate = df["Class"].mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(fraud_rate_by_hour.index, fraud_rate_by_hour.values, color=COLOR_FRAUD)
ax.axhline(overall_rate, color="black", linewidth=1, linestyle="--",
           label=f"Overall rate ({overall_rate:.3f}%)")

ax.set_xlabel("Hour of day (both days combined)")
ax.set_ylabel("Fraud rate (%)")
ax.set_title("Fraud rate by hour of day")
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Overall transaction volume shows a clear daily rhythm. It dips twice, once every night. This is expected for genuine cardholder activity. Fraud rate, however spikes overnight, it peaks around 02:00 and stays elevated into the early morning, then settles below the overall rate for most of the daytime hours. This is consistent with fraud being comparatively more common when legitimate transaction volume is at its lowest.

The linear relationship between `Time` and `Class` is negligible, it's approximately -0.012. This can be seen in the next section of this notebook. The chart above shows that the fraud rate varies with the hour of day, so there is a real signal, it's just not linear. Time will be kept as a feature that will be scaled inside the pipeline rather than dropped. Tree-based models can make use of the overnight pattern and a cyclical encoding of `hour_of_day` is noted as a possible refinement later. The engineered `hour_in_period` / `hour_of_day` columns used in this section are for EDA only and will not be part of the model's feature set.

## Correlation with the fraud label

In [ ]:
feature_cols = FEATURE_COLS
corr_with_class = df[feature_cols + ["Class"]].corr()["Class"].drop("Class")
corr_with_class = corr_with_class.sort_values()
corr_with_class.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = [COLOR_FRAUD if v > 0 else COLOR_LEGIT for v in corr_with_class]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson correlation with class")
ax.set_ylabel("Feature correlation with the fraud label")
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = df[v_cols + ["Time", "Amount", "Class"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, square=True,
            linewidths=0.3, cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Correlation matrix: V1-V28, Time, Amount, Class")
plt.tight_layout()
plt.show()

**Observation:** The `V` elements are mostly uncorrelated with each other. The heat map is flat except for the `Class` row and column. `Time` and `Amount` show very little correlation with the `V` elements as well.

Against the target, the strongest linear relationships are `V17`, `V14` and `V10`. `Time` and `Amount` barely register, with a correlation of less than 0.02.

## Screening all PCA components by Class

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(16, 20))
for ax, col in zip(axes.flat, v_cols):
    for cls, colour in CLASS_PALETTE.items():
        ax.hist(df.loc[df["Class"] == cls, col], bins=40, density=True,
                alpha=0.55, color=colour)
    ax.set_title(col, fontsize=10)
    ax.set_yticks([])

# One shared legend for the whole grid
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR_LEGIT, alpha=0.55),
                  plt.Rectangle((0, 0), 1, 1, color=COLOR_FRAUD, alpha=0.55)]
fig.legend(legend_handles, ["Legitimate", "Fraud"], loc="upper center",
           ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=11)
fig.suptitle("Distribution of each PCA component by Class",
             y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

From looking histograms we can see that most components show a lot of overlap between the two classes, but a few of the components like `V17`, `V14` and `V12` show the fraud distribution visibly shifted away from the legitimate one. This give an idea of which anonymised components are likely to matter the most.

# Examinining the most discriminative features

In [ ]:
abs_corr = corr_with_class.abs().sort_values(ascending=False)
top_features = abs_corr.head(6).index.tolist()
print(top_features)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, top_features):
    sns.boxplot(data=df, x="Class", y=col, hue="Class", palette=CLASS_PALETTE,
                showfliers=False, legend=False, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Legit", "Fraud"])
    ax.set_xlabel("")
    ax.set_title(f"{col}    (r = {corr_with_class[col]:.2f})")

fig.suptitle("The 6 features most correlated with fraud")
plt.tight_layout()
plt.show()

For all six features the fraud class's median and interquartile range sit clearly apart from the legitimate class's. This is clear visual confirmation that a classifier has a real signal to work with, even before modelling happens.

**Note:** Because the `V` components are anonymised PCA components, we can figure out that they seperate the classes, but we are unable to say why. A component with a large negative value for fraud can not be tracked back to a specific real-world attribute such as "location of transaction" or "rate of expenditure". That would only be possible with a dataset that isn't anonymised. Any feature importance analysis performed is therefore useful for building a good classifier, but limited when it comes to finding business explanations.

## Key findings

**The dataset is extremely imbalanced.**
after removing duplicates, fraud accounts for just 473 out of 283,726 transactions. That's 0.167%, roughly 1 in 600. The implications of this are:

- Accuracy is meaningless. A model that predicts "legitimate" for every row will score 99.8% while catching no fraud. The model will instead be evaluated with precision, recall, F1 and AUPRC.
- The train/test split must be stratified on `Class` so that the test set will have enough fraud cases to measure against.
- The imbalance needs direct treatment. If left untreated, most classifiers would optimise for overall accuracy, this would effectively ignore the minority class (fraud).

**Data quality is high.**
There are no missing values. 1,081 duplicate rows were found and 19 of these were fraud. They were dropped after checking their class balance, leaving 283,726 rows and 31 columns. All features are numeric.

**`Time` and `Amount` need scaling.**
`V1`-`V28` are already centred and on a broadly similar scale, `Time` and `Amount` however were outside the PCA step and would dominate distance and gradient based models, so StandardScaler will be applied.

**Fraud amounts are typically smaller.**
The median amount for fraud is 9.82 vs 22 for legitimate transactions. Even though both classes are heavily right skewed, `Amount` on its own is a weak discriminator (r = 0.006).

**Fraud has a non-linear time-of-day pattern.**
The fraud rates spike overnight and peak around 02:00 am, when the volume of legitimate transactions is at its lowest. Linear correlation with `Time` is negligible, but the overnight pattern is a real non-linear signal so `Time` is kept and scaled rather than dropped. A cyclical encoding of the hour is a possible refinement for later.

**A few PCA components carry most of the signal.**
The `V` components aren't correlated to each other. The ones with the strongest correlation to fraud are `V17`, `V14`, `V12`, `V10`, `V16`, `V3`, `V11` and `V4`.  On the other hand, `Time` and `Amount` barely register (|r| < 0.02). All 28 components are kept. There is no multicollinearity reason to drop any of them and no manual selection is done.

**Anonymised components.**
Since the PCA components (`V1`-`V28`) are anonymised, feature importance can be ranked but, we can't find out why the individual components matter. They can't be translated to business insights.

*Note regarding leakage:* in order to obtain a comprehensive picture, this EDA was performed on the entire de-duplicated dataset. The previously-mentioned feature signal observations are simply directional. No information from the test fold reaches the model because the modelling section re-derives them on the training fold, and every fitted transformation (scaling included) lives inside a `Pipeline` fit on training data, so no information from the test fold reaches the model.

## Train/Test Split

In [ ]:
# Create features and targe column
X = df[FEATURE_COLS]
y = df[TARGET_COL]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED
)

# Print train and test data shapes
X_train.shape, X_test.shape

In [ ]:
# Stratification check
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True)
}).mul(100).round(3)

## Baseline Model: Logistic Regression
A linear model is a good starting point because it is fast, simple and does not require much tuning. Its performance gives us a baseline that the later models, such as Random Forest, XGBoost and neural networks should aim to improve on.

The model is evaluated on the held-out test set using the same evaluation helper that the other models will use. This ensures that the test results can be compared fairly.

For now, the model is trained without any class balance correction. `class_weight="balanced"` will be introduced later to investigate how handling class imbalance affects the model's performance compared with the baseline.

In [ ]:
baseline_model = build_model_pipeline(
    LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    )
)

baseline_model.fit(X_train, y_train)

In [ ]:
baseline_metrics = evaluate_classifier(baseline_model, X_test, y_test)
pd.Series(baseline_metrics)

In [ ]:
plot_confusion_matrix(
    y_test,
    baseline_model.predict(X_test),
    title="Baseline logistic regression (test set)"
)

plt.show()

### Why accuracy is the wrong metric

In [ ]:
always_legit = np.zeros_like(y_test)

acc_naive = accuracy_score(y_test, always_legit)
acc_baseline = accuracy_score(y_test, baseline_model.predict(X_test))

print(f"'Always legit' accuracy:    {acc_naive:.4%}" )
print(f"Baseline model accuracy:    {acc_baseline:.4%}")

### Baseline model results
| Metric        |        Value |
|---------------|-------------:|
| Precision     |     0.879310 |
| Recall        |     0.536842 |
| F1            |     0.666667 |
| AUPRC         |     0.742974 |
| ROC-AUC       |     0.968538 |
| tn            | 56644.000000 |
| fp            |    7.0000000 |
| fn            |   44.0000000 |
| tp            |   51.0000000 |
| expected_cost | 4435.0000000 |



On the test set, there were **95 fraud cases out of 56,746 transactions**. The model correctly identified **51 fraud cases**, missed **44** and incorrectly flagged **7 legitimate transactions** as fraud.

**Accuracy is not a useful metric here.** An "always legitimate" model would achieve 99.83% accuracy, while the baseline achieves 99.91%. Despite the small difference, the two models are very different: the first catches no fraud, while the baseline catches more than half of the fraud cases. For this reason, accuracy will not be used to compare the models going forward.

**Precision is high, but recall is relatively low.** When the baseline flags a transaction as fraudulent, it is correct **88% of the time**. However, it only identifies **54% of all fraud cases**. Since missing a fraudulent transaction can be more costly than incorrectly flagging a legitimate one, **recall and AUPRC will be the main metrics used to compare the later models**.

The baseline therefore provides the **performance level that the later models should aim to improve on**, particularly in terms of recall and AUPRC.

### Random Forest

A Random Forest is a machine learning model made up of many decision trees. Each tree is trained using a random sample of the data and a random selection of features when making splits. Combining the predictions from many trees helps reduce the risk of relying too heavily on any single tree.

Random Forests make decisions using feature thresholds, which means they do not require features to be on the same scale. Therefore, the `StandardScaler` in the pipeline is not necessary for this model, but keeping it does not cause any problems.

`RandomizedSearchCV` samples 30 combinations of `n_estimators`, `max_depth`, `min_samples_leaf` and `max_features` using 3-fold cross-validation. Candidates are scored on **average precision (AUPRC)** rather than accuracy, since AUPRC is the appropriate metric for this imbalanced problem.

In [ ]:
# Instantiate random forest model
rf_pipeline = build_model_pipeline(
    RandomForestClassifier(random_state=RANDOM_SEED)
)

## Hyperparameter tuning
### Why These Random Forest Parameters Were Chosen

The Random Forest parameters were chosen to test different levels of model complexity and stability while taking into account the highly imbalanced fraud dataset.

- **`n_estimators: [200, 300, 400, 500, 600]`**
  This controls the number of decision trees in the forest. Using more trees generally makes the model's predictions more stable, although the improvement becomes smaller as more trees are added. Testing between 200 and 600 trees allows us to see whether increasing the size of the forest improves performance.

- **`max_depth: [2, 3, 5, 10, 20, 50]`**
  This controls how deep each decision tree can grow. Deeper trees can learn more complex patterns, but they can also overfit the training data. The range includes both shallow and deep trees so we can compare simpler models with more complex ones.

- **`min_samples_split: [5, 10, 20, 50, 100, 200]`**
  This controls the minimum number of samples required to split a node into smaller branches. Higher values prevent the trees from creating many very small branches, which can help reduce overfitting. A wider range was tested to find a suitable balance between learning useful patterns and keeping the trees from becoming too specific to the training data.

- **`min_samples_leaf: [5, 10, 20, 30]`**
  This controls the minimum number of samples that must be present in each leaf. This is particularly important for this dataset because fraud cases are rare. Requiring several observations per leaf helps prevent the model from making fraud probability estimates based on only a small number of transactions.

- **`max_features: ["sqrt", "log2", 0.3, 0.5]`**
  This controls how many features are considered when looking for the best split at each node. Using different subsets of features makes the trees more diverse, which is one of the main strengths of a Random Forest. These values allow us to test several levels of feature selection and determine which produces the best results.

`GridSearchCV` tests every combination of these parameter values using **3-fold cross-validation**. The models are ranked using **average precision (AUPRC)** rather than accuracy because the dataset contains far more legitimate transactions than fraudulent ones. This makes AUPRC a more useful measure of how well the model identifies fraud.

In [ ]:
# Parameters to tune
params = {
    "classifier__n_estimators": [200, 300, 400, 500, 600],
    "classifier__max_depth": [2, 3, 5, 10, 20, 50],
    "classifier__min_samples_split": [5, 10, 20, 50, 100, 200],
    "classifier__min_samples_leaf": [5, 10, 20, 30],
    "classifier__max_features": ["sqrt", "log2", 0.3, 0.5]
}

# Find best hyperparameters
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=params,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring="average_precision"
)

grid_search.fit(X_train, y_train)

In [ ]:
# Fit hypertuned model
rf_best = grid_search.best_estimator_
rf_best

In [ ]:
rf_metrics = evaluate_classifier(rf_best, X_test, y_test)
pd.Series(rf_metrics)